# DeliveryPulse — разведочный анализ

## TL;DR

Цель — воспроизводимо описать SLA, убыточность, маршруты, клиентов, события и автопарк на основе проверенного DuckDB warehouse и подготовить кандидатов для этапа 6. Данные синтетические; наблюдения ниже не являются причинными выводами и формальных статистических тестов здесь нет.

## Контекст, определения и ограничения

Анализ читает пять SQL-витрин, справочники и `warehouse_metadata`. Канонические KPI берутся из baseline SQL. OTD считается среди завершённых доставок, а групповая маржа — как `sum(delivery_profit) / sum(net_revenue)`. Денежная единица — RUB; бизнес-календарь — Europe/Moscow. `route_events` используется только для точечной детализации типов событий. Raw CSV и DuckDB не изменяются.

In [ ]:
from pathlib import Path

from delivery_pulse.analysis import run_eda
from delivery_pulse.analysis.reporting import hypothesis_candidates

PROJECT_ROOT = Path.cwd()
DATABASE = PROJECT_ROOT / "data" / "processed" / "delivery_pulse.duckdb"
OUTPUT_DIR = PROJECT_ROOT / "reports"
TOP_N = 10
MIN_GROUP_SIZE = 30

if not DATABASE.is_file():
    raise FileNotFoundError(
        "Сначала создайте data/processed/delivery_pulse.duckdb командой warehouse build"
    )

result = run_eda(
    DATABASE,
    OUTPUT_DIR,
    top_n=TOP_N,
    min_group_size=MIN_GROUP_SIZE,
)
print(f"Database: {result.context.database}")
print(f"Project version: {result.context.metadata['project_version']}")
profile = result.context.metadata["profile"]
seed = result.context.metadata["seed"]
print(f"Profile / seed: {profile} / {seed}")
start_date = result.context.metadata["start_date"]
months = result.context.metadata["months"]
print(f"Start / months: {start_date} / {months}")
print(f"Warehouse rows: {result.context.row_counts}")

## Общий обзор

In [ ]:
baseline = result.context.baseline
for key, value in baseline.items():
    print(f"{key}: {value}")

## Временная динамика

Таблица использует бизнес-месяц Europe/Moscow. Временное совпадение изменений не трактуется как причина.

In [ ]:
result.tables["monthly"].head(12)

## Маршруты и клиенты

Процентные рейтинги применяют минимальный размер группы; размер выборки показан рядом.

In [ ]:
result.tables["route_loss_ranking"].head(TOP_N)

In [ ]:
result.tables["customer_loss_ranking"].head(TOP_N)

## События и задержки

Сравнения `with/without` описательные и не контролируют маршрутный, клиентский или временной состав.

In [ ]:
result.tables["event_comparisons"]

## Убыточность и автопарк

Нормированные поломки рассматриваются только вместе с пробегом и часами экспозиции.

In [ ]:
result.tables["profitability_segments"]

In [ ]:
result.tables["vehicle_breakdown_ranking"].head(TOP_N)

## Standard и express

Это абсолютные описательные показатели без проверки значимости.

In [ ]:
result.tables["priority"]

## Графики и отчёт

In [ ]:
print(f"Markdown report: {result.report_path}")
for name, path in sorted(result.figures.items()):
    print(f"{name}: {path}")

## Кандидаты для этапа 6

Ниже только дизайн будущих проверок; сами тесты на этом этапе не выполняются.

In [ ]:
for number, candidate in enumerate(hypothesis_candidates(), start=1):
    print(f"{number}. {candidate['title']}")
    print(f"   Метрика: {candidate['metric']}")
    print(f"   Минимум: {candidate['minimum']}")

## Выводы и границы интерпретации

Фактические числа, описательные наблюдения, возможные объяснения и гипотезы разделены в сгенерированном `reports/eda_summary.md`. Следующий этап должен заранее зафиксировать выборки, методы, контроль факторов и критерии решений. Синтетическая конструкция набора, шум, пересекающиеся распределения и сегментные эффекты ограничивают перенос выводов за пределы проекта.